In [2]:
import pandas as pd
from Levenshtein import distance as levenshtein_distance

def gender_dictionary(df):
    name_gender_counts = df.groupby("firstname")["gender"].value_counts().unstack(fill_value=0)
    name_gender_dict = name_gender_counts.to_dict(orient="index")

    final_name_gender_dict = {}

    for name, counts in name_gender_dict.items():
        total = sum(counts.values())  
        if total > 0:
            max_gender = max(counts, key=counts.get)
            max_ratio = counts[max_gender] / total

            if max_ratio >= 0.8:
                final_name_gender_dict[name] = max_gender
            else:
                final_name_gender_dict[name] = "uncertain"
        else:
            final_name_gender_dict[name] = "uncertain"
            
    return final_name_gender_dict

def check_marital_status(text, max_distance=2):
    if pd.isna(text) or not isinstance(text, str) or text.strip() == "":
        return None
    
    text = text.lower().strip()
    
    female_markers = ["vrouw", "vr", "weduwe", "wwe", "echtgenoot", "dochter", "moeder", "zuster"]
    male_markers = ["zoon", "vader", "broeder"]
    
    for marker in female_markers:
        if marker in text:
            return "woman"
    
    for marker in male_markers:
        if marker in text:
            return "man"
    
    for word in text.split():
        for marker in female_markers:
            if levenshtein_distance(word, marker) <= max_distance:
                return "woman"
    
    for word in text.split():
        for marker in male_markers:
            if levenshtein_distance(word, marker) <= max_distance:
                return "man"
                
    return None

def check_title(title, max_distance=2):
    if pd.isna(title) or not isinstance(title, str) or title.strip() == "":
        return None
    
    title = title.lower().strip()
    
    # Keep existing title markers
    if "madame" in title or "madamme" in title or "mme" in title:
        return "woman"
    
    if "mr" in title:
        return "man"
    
    # Add the same markers as used in marital status
    female_markers = ["vrouw", "vr", "weduwe", "wwe", "echtgenoot", "dochter", "moeder", "zuster"]
    male_markers = ["zoon", "vader", "broeder"]
    
    for marker in female_markers:
        if marker in title:
            return "woman"
    
    for marker in male_markers:
        if marker in title:
            return "man"
    
    for word in title.split():
        for marker in female_markers:
            if levenshtein_distance(word, marker) <= max_distance:
                return "woman"
    
    for word in title.split():
        for marker in male_markers:
            if levenshtein_distance(word, marker) <= max_distance:
                return "man"
    
    return None

def predict_gender(row, name_dict, max_distance=2):
    # Initialize result values
    gender_result = "unknown"
    matching_name = None
    distance_value = None
    
    # Look for marital status field - check if it exists in row
    marital_status_field = None
    for field in row.index:
        if 'MARITALSTATUS' in field:
            marital_status_field = field
            break
    
    # Check marital status if the field exists
    if marital_status_field and not pd.isna(row[marital_status_field]):
        marital_gender = check_marital_status(row[marital_status_field])
        if marital_gender:
            return marital_gender, None, None  # No name match in this case
    
    # Look for title field - check if it exists in row
    title_field = None
    for field in row.index:
        if 'TITLE' in field:
            title_field = field
            break
    
    # Check title if the field exists
    if title_field and not pd.isna(row[title_field]):
        title_gender = check_title(row[title_field])
        if title_gender:
            return title_gender, None, None  # No name match in this case
    
    # Use the specific P_FIRSTNAME column
    firstname = None
    
    # Check if P_FIRSTNAME exists and is not NaN
    if 'P_FIRSTNAME' in row and not pd.isna(row['P_FIRSTNAME']):
        # Extract first word from firstname
        firstname_parts = str(row['P_FIRSTNAME']).split()
        if firstname_parts:
            firstname = firstname_parts[0]
    
    if pd.isna(firstname) or not isinstance(firstname, str) or firstname.strip() == "":
        return "unknown", None, None
    
    # Check if firstname is just an initial (single letter)
    if len(firstname.strip()) == 1:
        return "unknown", None, None
    
    # Exact match
    if firstname in name_dict:
        return name_dict[firstname], firstname, 0
    
    # Try Levenshtein matches with increasing distance
    best_match = None
    min_distance = float('inf')
    
    for existing_name, gender in name_dict.items():
        current_distance = levenshtein_distance(firstname.lower(), existing_name.lower())
        if current_distance <= max_distance and current_distance < min_distance:
            min_distance = current_distance
            best_match = existing_name
            gender_result = gender
    
    if best_match:
        return gender_result, best_match, min_distance
    
    return "unknown", None, None

def main():
    try:
        # Read the first Excel file with names and genders
        print("Reading names database...")
        df_names = pd.read_excel("../../../data/H4/normalisering/SOSA_Datatabank_voornaam en sekse.xlsx")
        
        # Rename columns to standardized names
        print("Processing names data...")
        df_names = df_names.rename(columns={
            "Data-\nbank\nCode\nNEW": "DB_code", 
            "Voor-\nnamen": "names", 
            "Sterfte\nJaar": "year_death", 
            "Geslacht": "gender"
        })
        
        # Extract first name from full names
        df_names["firstname"] = df_names["names"].str.split().str[0]
        df_names = df_names[~df_names["firstname"].str.contains(r"\.", na=False)]
        df_names = df_names[df_names["gender"].isin([1, 2])]
        df_names["gender"] = df_names["gender"].replace({1: "man", 2: "woman"})
        
        # Create gender dictionary
        name_gender_dict = gender_dictionary(df_names)
        
        # Read the second Excel file with persons data
        print("Reading persons data...")
        df_persons = pd.read_excel("../../../data/H4/normalisering/machinaal_test.xlsx")
        
        # Extract first word from firstname
        df_persons['firstname_firstword'] = df_persons["P_FIRSTNAME"].str.extract(r'^([\wÀ-ÿ]+)')
        
        # Apply gender inference function to each row - now with expanded return values
        print("Inferring genders...")
        
        # Create empty lists to store the results
        genders = []
        matched_names = []
        distances = []
        
        # Process each row
        for _, row in df_persons.iterrows():
            gender, matched_name, distance = predict_gender(row, name_gender_dict, max_distance=2)
            genders.append(gender)
            matched_names.append(matched_name)
            distances.append(distance)
        
        # Add results to the dataframe
        df_persons["gender_inferred"] = genders
        df_persons["firstname_match"] = matched_names
        df_persons["levenshtein_distance"] = distances
        
        # Save the result
        print("Saving results...")
        df_persons.to_excel("../../../data/H4/normalisering/machinaal_test_gender.xlsx", index=False)
        print("Gender inference completed successfully!")
        
        # Print some statistics
        total_records = len(df_persons)
        exact_matches = (df_persons["levenshtein_distance"] == 0).sum()
        distance_1_matches = (df_persons["levenshtein_distance"] == 1).sum()
        distance_2_matches = (df_persons["levenshtein_distance"] == 2).sum()
        no_matches = df_persons["firstname_match"].isna().sum()
        
        print(f"\nStatistics:")
        print(f"Total records: {total_records}")
        print(f"Exact matches: {exact_matches} ({exact_matches/total_records*100:.2f}%)")
        print(f"Levenshtein distance 1 matches: {distance_1_matches} ({distance_1_matches/total_records*100:.2f}%)")
        print(f"Levenshtein distance 2 matches: {distance_2_matches} ({distance_2_matches/total_records*100:.2f}%)")
        print(f"No name matches: {no_matches} ({no_matches/total_records*100:.2f}%)")
        
    except Exception as e:
        print(f"Error occurred: {str(e)}")
        # Print more detailed error information
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

Reading names database...
Processing names data...
Reading persons data...
Inferring genders...
Saving results...
Gender inference completed successfully!

Statistics:
Total records: 14179
Exact matches: 5693 (40.15%)
Levenshtein distance 1 matches: 1247 (8.79%)
Levenshtein distance 2 matches: 557 (3.93%)
No name matches: 6682 (47.13%)
